# Graphs

In [1]:
# !python dxf2chunks.py --input_fold 'data'

# Visualize chunks

In [2]:
import json
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
import numpy as np
from matplotlib import colormaps

def visualize_geojson_layouts(geojson_path, max_layouts=None):
    # === 0. SAFETY CHECKS ===
    if not os.path.exists(geojson_path):
        print(f"[ERROR] File not found: {geojson_path}")
        return

    # Check if file is empty
    if os.path.getsize(geojson_path) == 0:
        print(f"[ERROR] The file '{geojson_path}' is empty (0 bytes).")
        print("       Check your pipeline generation step. Did the DXF extraction fail?")
        return

    # === 1. LOAD AND GROUP GEOJSON DATA ===
    print(f"Loading {geojson_path}...")
    temp_grouped = {} 

    try:
        with open(geojson_path, 'r') as f:
            data = json.load(f) # Load the whole FeatureCollection
    except json.JSONDecodeError as e:
        print(f"[ERROR] Could not decode JSON in '{geojson_path}'.")
        print(f"       Details: {e}")
        return

    # Check if 'features' key exists (in case it saved an empty list [])
    if 'features' not in data:
        print(f"[WARN] No 'features' key found in JSON. Is this a FeatureCollection?")
        return

    for feature in data['features']:
        props = feature['properties']
        geom = feature['geometry']
        
        lid = props['layout_id']
        cid = props['chunk_id']
        
        if not geom or 'coordinates' not in geom or not geom['coordinates']:
            continue
        
        # Extract the exterior ring coordinates
        poly_coords = geom['coordinates'][0]

        if lid not in temp_grouped:
            temp_grouped[lid] = {}
        if cid not in temp_grouped[lid]:
            temp_grouped[lid][cid] = []
        
        temp_grouped[lid][cid].append(poly_coords)

    # === 2. CONVERT TO VISUALIZATION FORMAT ===
    layout_ids = sorted(temp_grouped.keys())
    
    if not layout_ids:
        print("[WARN] No valid layouts found to visualize.")
        return

    if max_layouts:
        layout_ids = layout_ids[:max_layouts]

    for lid in layout_ids:
        chunk_dict = temp_grouped[lid]
        
        # Prepare data containers
        chunks_for_plot = []
        all_layout_coords = []

        # Convert dictionary {cid: [poly...]} to list [{'chunk_id': cid, 'coords_list': [...]}]
        sorted_cids = sorted(chunk_dict.keys())
        for cid in sorted_cids:
            polys = chunk_dict[cid]
            chunks_for_plot.append({
                'chunk_id': cid,
                'coords_list': polys
            })
            all_layout_coords.extend(polys)

        n_chunks = len(chunks_for_plot)

        # === 3. PLOTTING LOGIC ===
        
        # Calculate Global Bounds
        all_polys_flat = [pt for poly in all_layout_coords for pt in poly]
        if not all_polys_flat:
            continue
            
        all_points = np.array(all_polys_flat)
        min_x, min_y = all_points.min(axis=0)
        max_x, max_y = all_points.max(axis=0)
        
        # Add 5% padding
        pad_x = (max_x - min_x) * 0.05
        pad_y = (max_y - min_y) * 0.05
        global_xlim = (min_x - pad_x, max_x + pad_x)
        global_ylim = (min_y - pad_y, max_y + pad_y)

        # Dynamic grid setup
        cols = min(5, n_chunks + 1)
        rows = (n_chunks // cols) + 2

        fig = plt.figure(figsize=(5 * cols, 5 * rows))
        fig.suptitle(f"Layout {lid} → {len(all_layout_coords)} Regions, {n_chunks} Chunk{'s' if n_chunks != 1 else ''}",
                     fontsize=20, y=0.95)
        
        # Updated colormap retrieval
        cmap = colormaps.get_cmap("tab20")

        # --- A. Full Layout Overview ---
        ax_full = fig.add_subplot(rows, cols, 1)
        
        patches = []
        colors = []
        
        for chunk in chunks_for_plot:
            color = cmap(chunk['chunk_id'] % 20)
            for coords in chunk['coords_list']:
                if len(coords) >= 3:
                    patches.append(MplPolygon(np.array(coords), closed=True))
                    colors.append(color)

        if patches:
            collection = PatchCollection(patches, facecolor=colors, edgecolor='black', linewidth=0.5, alpha=0.9)
            ax_full.add_collection(collection)
        
        ax_full.set_xlim(global_xlim)
        ax_full.set_ylim(global_ylim)
        ax_full.set_aspect('equal')
        ax_full.set_title("Full Layout (Chunk Coloring)", fontsize=14, pad=20)
        ax_full.axis('off')

        # --- B. Individual Chunks ---
        for i, chunk in enumerate(chunks_for_plot):
            ax = fig.add_subplot(rows, cols, i + cols + 1)
            
            # Ghost background
            ghost_patches = []
            for other_chunk in chunks_for_plot:
                if other_chunk['chunk_id'] != chunk['chunk_id']:
                    for coords in other_chunk['coords_list']:
                        if len(coords) >= 3:
                            ghost_patches.append(MplPolygon(np.array(coords), closed=True))
            if ghost_patches:
                ghost_col = PatchCollection(ghost_patches, facecolor='#f0f0f0', edgecolor='#e0e0e0', linewidth=0.5)
                ax.add_collection(ghost_col)

            # Active Chunk
            chunk_patches = [MplPolygon(np.array(coords), closed=True) for coords in chunk['coords_list'] if len(coords) >= 3]

            if chunk_patches:
                color = cmap(chunk['chunk_id'] % 20)
                collection = PatchCollection(chunk_patches, facecolor=color, edgecolor='black', linewidth=0.7, alpha=0.9)
                ax.add_collection(collection)

            ax.set_xlim(global_xlim)
            ax.set_ylim(global_ylim)
            ax.set_aspect('equal')
            ax.set_title(f"Chunk {chunk['chunk_id']}", fontsize=12)
            ax.axis('off')

        # Hide unused subplots
        total_plots = 1 + n_chunks
        for j in range(total_plots, rows * cols):
            fig.add_subplot(rows, cols, j + 1).axis('off')

        plt.tight_layout(rect=[0, 0, 1, 0.94])
        plt.subplots_adjust(top=0.92, hspace=0.3, wspace=0.1)
        # plt.savefig(f'demo_chunk_{lid}.png')
        plt.show()

if __name__ == "__main__":
    # Update this path to where your file is actually located
    visualize_geojson_layouts("chunked_data/冷冻机房0327_t3.geojson", max_layouts=None)

[ERROR] File not found: chunked_data/冷冻机房0327_t3.geojson


# Image-DXF transformations

In [3]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
import cv2

# ==========================================
# 1. HELPER FUNCTIONS (From your original script)
# ==========================================

def load_transform(transform_path):
    with open(transform_path, 'r') as f:
        return json.load(f)

def transform_polygon_to_pixels(polygon_coords, transform):
    """
    Converts a list of [x, y] world coordinates to pixel coordinates.
    """
    coords = np.array(polygon_coords)
    
    # 1. Shift by Min X
    pixel_x = (coords[:, 0] - transform['min_x']) * transform['pixel_per_unit']
    
    # 2. Shift by Max Y and FLIP (because image Y=0 is top, DXF Y=0 is bottom)
    pixel_y = (transform['max_y'] - coords[:, 1]) * transform['pixel_per_unit']
    
    # 3. Stack and Round
    return np.column_stack((pixel_x, pixel_y)).astype(int)

# ==========================================
# 2. DATAFRAME CREATION LOGIC
# ==========================================

def create_layout_dataframe(geojson_path, transform_path):
    """
    Parses GeoJSON and Transform file to create a Pandas DataFrame.
    """
    # Load external files
    try:
        with open(geojson_path, 'r') as f:
            geojson_data = json.load(f)
        transform = load_transform(transform_path)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return pd.DataFrame()

    # Derive DXF filename from the geojson filename (assuming convention)
    # e.g., "chunked_data/Floor plan.geojson" -> "Floor plan.dxf"
    base_name = os.path.splitext(os.path.basename(geojson_path))[0]
    dxf_filename = f"{base_name}.dxf"

    rows = []

    print(f"[INFO] Processing {len(geojson_data['features'])} features...")

    for feature in geojson_data['features']:
        props = feature['properties']
        geom = feature['geometry']

        # Extract IDs
        layout_id = props.get('layout_id')
        chunk_id = props.get('chunk_id')

        # Extract and Transform Geometry
        # Note: GeoJSON polygons are usually [[[x,y], [x,y]...]] (list of rings)
        # We take the first ring (exterior)
        if geom and 'coordinates' in geom:
            raw_coords = geom['coordinates'][0]
            
            # Apply the transformation logic to get pixel coordinates
            pixel_poly = transform_polygon_to_pixels(raw_coords, transform)

            rows.append({
                'dxf_file': dxf_filename,
                'layout_id': layout_id,
                'chunk_id': chunk_id,
                'chunks': pixel_poly  # Storing the numpy array of pixels
            })

    # Create DataFrame
    df = pd.DataFrame(rows)
    return df

# ==========================================
# 3. VISUALIZATION FROM DATAFRAME
# ==========================================
def plot_multiple_chunks(df, image_path, target_layouts, target_chunks):
    """
    Queries the DataFrame and plots multiple specific chunks/layouts on the image.
    Accepts lists for target_layouts and target_chunks.
    """
    
    # 0. NORMALIZE INPUTS TO LISTS
    # This ensures the code works even if you pass a single integer (e.g. 0 instead of [0])
    if not isinstance(target_layouts, (list, tuple, np.ndarray)):
        target_layouts = [target_layouts]
    if not isinstance(target_chunks, (list, tuple, np.ndarray)):
        target_chunks = [target_chunks]

    # 1. QUERY THE DATAFRAME
    # Filter using .isin() to match any value in the provided lists
    subset = df[
        (df['layout_id'].isin(target_layouts)) & 
        (df['chunk_id'].isin(target_chunks))
    ]

    if subset.empty:
        print(f"[WARN] No data found for Layouts {target_layouts}, Chunks {target_chunks}")
        return

    # 2. Load Image
    try:
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except Exception as e:
        print(f"[ERROR] Could not load image: {e}")
        return

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(img)

    # 3. Plot all polygons associated with the query
    pixel_polygons = []
    
    # Generate distinct colors for the number of unique items we found
    # We create a unique key for every (layout, chunk) pair to assign a color
    unique_pairs = subset[['layout_id', 'chunk_id']].drop_duplicates()
    cmap = plt.get_cmap('tab20') # 'tab20' has 20 distinct colors
    
    # Create a dictionary mapping (layout_id, chunk_id) -> Color
    color_map = {}
    for idx, (l_id, c_id) in enumerate(zip(unique_pairs['layout_id'], unique_pairs['chunk_id'])):
        color_map[(l_id, c_id)] = cmap(idx % 20)

    for _, row in subset.iterrows():
        poly_points = row['chunks']
        pixel_polygons.append(poly_points)
        
        # Determine color based on this row's IDs
        current_color = color_map.get((row['layout_id'], row['chunk_id']), 'lime')
        
        patch = MplPolygon(
            poly_points, 
            closed=True, 
            facecolor=current_color, 
            edgecolor='white', 
            alpha=0.6,
            linewidth=2
        )
        ax.add_patch(patch)

    # 4. Zoom camera to the specific chunks
    if pixel_polygons:
        all_points = np.vstack(pixel_polygons)
        min_x, min_y = all_points.min(axis=0)
        max_x, max_y = all_points.max(axis=0)
        
        pad = 50
        ax.set_xlim(min_x - pad, max_x + pad)
        ax.set_ylim(max_y + pad, min_y - pad) # Invert Y for image plotting
    
    ax.set_title(f"Multi-Chunk Query: Layouts {target_layouts} | Chunks {target_chunks}")
    plt.show()

# ==========================================
# 4. EXECUTION
# ==========================================
if __name__ == "__main__":
    
    # File Paths
    GEOJSON_FILE = "chunked_data/冷冻机房0327_t3.geojson"
    TRANSFORM_FILE = "images/冷冻机房0327_t3_transform.json"
    IMAGE_FILE   = "images/冷冻机房0327_t3.png"

    # 1. Create the DataFrame
    df = create_layout_dataframe(GEOJSON_FILE, TRANSFORM_FILE)

    # # 2. Inspect the DataFrame
    # print("\n--- DataFrame Head ---")
    # print(df.head())
    
    # print("\n--- DataFrame Info ---")
    # print(df.info())

    # # 3. Example: How to query specific chunks
    # # Let's say we want Layout 0, Chunk 20
    # print("\n--- Querying Layout 0, Chunk 20 ---")
    # query_result = df[(df['layout_id'] == 0) & (df['chunk_id'] == 20)]
    # print(f"Found {len(query_result)} polygon(s).")

    # # 4. Visualize based on the query
    # # (Only runs if the image file actually exists locally)
    # if os.path.exists(IMAGE_FILE) and not df.empty:
    #     plot_multiple_chunks(
    #         df, 
    #         IMAGE_FILE, 
    #         target_layouts=[0], 
    #         target_chunks=[1, 5, 8]
    #     )
    #     # plot_chunk_from_df(df, IMAGE_FILE, target_layout=0, target_chunk=20)

Error loading files: [Errno 2] No such file or directory: 'chunked_data/冷冻机房0327_t3.geojson'


In [4]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt # Needed for colormap
import matplotlib.cm as cm 
import numpy as np

def get_chunk_pillow_images(df, image_path, target_layouts, target_chunks, padding=50):
    """
    Queries the DataFrame, calculates the bounding box of specific chunks,
    and returns TWO Pillow images:
    1. The raw cropped region (clean).
    2. The cropped region with segmentation overlay.

    Returns:
        (PIL.Image, PIL.Image): Tuple (raw_crop, overlay_crop). 
                                Returns (None, None) if no data found.
    """
    
    # 1. NORMALIZE INPUTS
    if not isinstance(target_layouts, (list, tuple, np.ndarray)):
        target_layouts = [target_layouts]
    if not isinstance(target_chunks, (list, tuple, np.ndarray)):
        target_chunks = [target_chunks]

    # 2. QUERY DATAFRAME
    subset = df[
        (df['layout_id'].isin(target_layouts)) & 
        (df['chunk_id'].isin(target_chunks))
    ]

    if subset.empty:
        print(f"[WARN] No data found for Layouts {target_layouts}, Chunks {target_chunks}")
        return None, None

    # 3. LOAD IMAGE
    try:
        # Load as RGBA to handle transparency composition
        base_img = Image.open(image_path).convert("RGBA")
    except Exception as e:
        print(f"[ERROR] Could not load image: {e}")
        return None, None

    # 4. CREATE OVERLAY LAYER
    overlay = Image.new("RGBA", base_img.size, (255, 255, 255, 0))
    draw = ImageDraw.Draw(overlay)

    # Setup Colors
    unique_pairs = subset[['layout_id', 'chunk_id']].drop_duplicates()
    cmap = plt.get_cmap('tab20')
    color_map = {}
    
    for idx, (l_id, c_id) in enumerate(zip(unique_pairs['layout_id'], unique_pairs['chunk_id'])):
        rgba_float = cmap(idx % 20)
        rgb_int = tuple(int(c * 255) for c in rgba_float[:3])
        # Alpha 128 = ~50% transparency
        color_map[(l_id, c_id)] = rgb_int + (128,)

    # 5. DRAW POLYGONS & CALCULATE BOUNDS
    all_points_for_crop = []

    for _, row in subset.iterrows():
        poly_arr = row['chunks']
        poly_tuples = [tuple(pt) for pt in poly_arr]
        
        fill_color = color_map.get((row['layout_id'], row['chunk_id']), (0, 255, 0, 128))
        
        # Draw on the overlay layer
        draw.polygon(poly_tuples, fill=fill_color, outline="white")
        
        all_points_for_crop.append(poly_arr)

    # Create the Combined version
    combined_img = Image.alpha_composite(base_img, overlay)

    # 6. CROP BOTH IMAGES
    if all_points_for_crop:
        all_points = np.vstack(all_points_for_crop)
        
        min_x, min_y = all_points.min(axis=0)
        max_x, max_y = all_points.max(axis=0)

        width, height = base_img.size
        
        # Calculate Box with padding
        left = max(0, min_x - padding)
        top = max(0, min_y - padding)
        right = min(width, max_x + padding)
        bottom = min(height, max_y + padding)
        
        crop_box = (left, top, right, bottom)

        # A. Crop the Raw Base Image (Convert to RGB to drop Alpha channel)
        raw_crop = base_img.crop(crop_box).convert("RGB")
        
        # B. Crop the Combined Image
        overlay_crop = combined_img.crop(crop_box).convert("RGB")
        
        return raw_crop, overlay_crop
    
    # Fallback if no points found (return full images)
    return base_img.convert("RGB"), combined_img.convert("RGB")

raw_img, overlay_img = get_chunk_pillow_images(
    df, 
    IMAGE_FILE, 
    target_layouts=[0], 
    target_chunks=[0]
)

KeyError: 'layout_id'

In [ ]:
# overlay_img
# df.head(20)
df[['layout_id', 'chunk_id']].drop_duplicates()

In [ ]:
def get_layout_pillow_image(df, image_path, target_layouts, padding=50):
    """
    Queries the DataFrame for specific layout(s), calculates the bounding box 
    of ALL chunks within those layouts, and returns the raw cropped Pillow image.
    
    No overlays are applied.

    Args:
        df (pd.DataFrame): The dataframe with polygon data.
        image_path (str): Path to the source image.
        target_layouts (int or list): Layout ID(s) to visualize.
        padding (int): Pixels of padding around the cropped area.

    Returns:
        PIL.Image: The clean cropped image of the layout region.
                   Returns None if no data found.
    """
    
    # 1. NORMALIZE INPUTS
    if not isinstance(target_layouts, (list, tuple, np.ndarray)):
        target_layouts = [target_layouts]

    # 2. QUERY DATAFRAME (Filter by Layout ID only)
    subset = df[df['layout_id'].isin(target_layouts)]

    if subset.empty:
        print(f"[WARN] No data found for Layouts {target_layouts}")
        return None

    # 3. CALCULATE BOUNDING BOX
    # We don't need to iterate for drawing, just stack all points to find the extent.
    all_points_list = subset['chunks'].tolist()
    
    if not all_points_list:
        return None

    # Stack every polygon point from the entire layout into one massive array
    all_points = np.vstack(all_points_list)
    
    min_x, min_y = all_points.min(axis=0)
    max_x, max_y = all_points.max(axis=0)

    # 4. LOAD IMAGE
    try:
        # We don't strictly need RGBA here since we aren't compositing transparency,
        # but consistency helps. We will return RGB.
        base_img = Image.open(image_path)
    except Exception as e:
        print(f"[ERROR] Could not load image: {e}")
        return None

    # 5. CROP
    width, height = base_img.size
    
    left = max(0, min_x - padding)
    top = max(0, min_y - padding)
    right = min(width, max_x + padding)
    bottom = min(height, max_y + padding)
    
    # Crop and ensure RGB format
    cropped_img = base_img.crop((left, top, right, bottom)).convert("RGB")
    
    return cropped_img

layout_image = get_layout_pillow_image(
    df, 
    IMAGE_FILE, 
    target_layouts=[1], 
    padding=50
)

In [ ]:
# layout_image

In [ ]:
chunks = df[(df['layout_id']==0) & (df['chunk_id']==0)]['chunks'].values
chunks[1]

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
import cv2

# ==========================================
# 1. HELPER FUNCTIONS
# ==========================================

def load_transform(transform_path):
    with open(transform_path, 'r') as f:
        return json.load(f)

def transform_polygon_to_pixels(polygon_coords, transform):
    coords = np.array(polygon_coords)
    # 1. Shift by Min X
    pixel_x = (coords[:, 0] - transform['min_x']) * transform['pixel_per_unit']
    # 2. Shift by Max Y and FLIP
    pixel_y = (transform['max_y'] - coords[:, 1]) * transform['pixel_per_unit']
    # 3. Stack and Round
    return np.column_stack((pixel_x, pixel_y)).astype(int)

# ==========================================
# 2. DATAFRAME CREATION LOGIC
# ==========================================

def create_layout_dataframe(geojson_path, transform_path):
    try:
        with open(geojson_path, 'r') as f:
            geojson_data = json.load(f)
        transform = load_transform(transform_path)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return pd.DataFrame()

    base_name = os.path.splitext(os.path.basename(geojson_path))[0]
    dxf_filename = f"{base_name}.dxf"

    rows = []
    print(f"[INFO] Processing {len(geojson_data['features'])} features...")

    for feature in geojson_data['features']:
        props = feature['properties']
        geom = feature['geometry']

        layout_id = props.get('layout_id')
        chunk_id = props.get('chunk_id')

        if geom and 'coordinates' in geom:
            # Taking the exterior ring
            raw_coords = geom['coordinates'][0]
            pixel_poly = transform_polygon_to_pixels(raw_coords, transform)

            rows.append({
                'dxf_file': dxf_filename,
                'layout_id': layout_id,
                'chunk_id': chunk_id,
                'chunks': pixel_poly 
            })

    return pd.DataFrame(rows)

# ==========================================
# 3. NEW: SAVE MASKS TO JSON
# ==========================================

class NumpyEncoder(json.JSONEncoder):
    """ Special json encoder for numpy types """
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

def save_masks_to_json(df, output_path):
    """
    Converts the DataFrame into a list of dictionaries and saves as JSON.
    Includes Bounding Box (bbox) calculation for convenience.
    """
    mask_data = []

    print(f"[INFO] Converting {len(df)} masks to JSON format...")

    for _, row in df.iterrows():
        pixel_poly = row['chunks'] # This is a numpy array [[x,y], [x,y]]
        
        # Calculate Bounding Box [x, y, width, height] for metadata
        x, y, w, h = cv2.boundingRect(pixel_poly)
        
        mask_entry = {
            "layout_id": row['layout_id'],
            "chunk_id": row['chunk_id'],
            "bbox": [x, y, w, h], # x_min, y_min, width, height
            "segmentation": pixel_poly # Encoder will handle tolist()
        }
        mask_data.append(mask_entry)

    # Save to file using the custom encoder to handle numpy numbers/arrays
    with open(output_path, 'w') as f:
        json.dump(mask_data, f, cls=NumpyEncoder, indent=4)
    
    print(f"[SUCCESS] Masks saved to {output_path}")

# ==========================================
# 4. VISUALIZATION (UNCHANGED)
# ==========================================
def plot_multiple_chunks(df, image_path, target_layouts, target_chunks):
    if not isinstance(target_layouts, (list, tuple, np.ndarray)):
        target_layouts = [target_layouts]
    if not isinstance(target_chunks, (list, tuple, np.ndarray)):
        target_chunks = [target_chunks]

    subset = df[
        (df['layout_id'].isin(target_layouts)) & 
        (df['chunk_id'].isin(target_chunks))
    ]

    if subset.empty:
        print(f"[WARN] No data found.")
        return

    try:
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except Exception as e:
        print(f"[ERROR] Could not load image: {e}")
        return

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(img)

    pixel_polygons = []
    unique_pairs = subset[['layout_id', 'chunk_id']].drop_duplicates()
    cmap = plt.get_cmap('tab20')
    
    color_map = {}
    for idx, (l_id, c_id) in enumerate(zip(unique_pairs['layout_id'], unique_pairs['chunk_id'])):
        color_map[(l_id, c_id)] = cmap(idx % 20)

    for _, row in subset.iterrows():
        poly_points = row['chunks']
        pixel_polygons.append(poly_points)
        current_color = color_map.get((row['layout_id'], row['chunk_id']), 'lime')
        
        patch = MplPolygon(
            poly_points, closed=True, facecolor=current_color, 
            edgecolor='white', alpha=0.6, linewidth=2
        )
        ax.add_patch(patch)

    if pixel_polygons:
        all_points = np.vstack(pixel_polygons)
        min_x, min_y = all_points.min(axis=0)
        max_x, max_y = all_points.max(axis=0)
        pad = 50
        ax.set_xlim(min_x - pad, max_x + pad)
        ax.set_ylim(max_y + pad, min_y - pad)
    
    plt.show()

# ==========================================
# 5. EXECUTION
# ==========================================
if __name__ == "__main__":
    
    # File Paths
    GEOJSON_FILE = "chunked_data/冷冻机房0327_t3.geojson"
    TRANSFORM_FILE = "images/冷冻机房0327_t3_transform.json"
    IMAGE_FILE   = "images/冷冻机房0327_t3.png"
    
    # New Output File
    OUTPUT_MASK_JSON = "chunked_data/pixel_masks.json"

    # 1. Create the DataFrame
    df = create_layout_dataframe(GEOJSON_FILE, TRANSFORM_FILE)

    if not df.empty:
        # 2. Save the Mask JSON
        save_masks_to_json(df, OUTPUT_MASK_JSON)

        # 3. Visualize to verify
        if os.path.exists(IMAGE_FILE):
            print("\n--- Visualizing a sample to verify ---")
            plot_multiple_chunks(df, IMAGE_FILE, target_layouts=[0], target_chunks=[0,1,2,3,4,5])

In [ ]:
# from PIL import Image
# Image.open("images/冷冻机房0327_t3_OVERLAY.png")